In [ ]:
# ======================================================================================
# FINAL ECG RECONSTRUCTION V6
#
# Robust version:
#   - uses existing browser annotations
#   - NO global dynamic programming
#   - interpolates manual anchor trajectory
#   - locally snaps only a few pixels toward dark ECG ink
#   - V4 retained for every lead marked Keep V4
#
# INPUT
#   LOCAL_USER_HOME/Downloads/ecg_anchor_annotations*.json
#   digitization_qc_v4/X_gaze_v4.npy
#   manual_digitization_qc/manual_geometry.json
#
# OUTPUT
#   final_digitisation_v6/
#       X_gaze_final.npy
#       FINAL_OVERLAY_<ECG>.png
#       FINAL_SIGNAL_<ECG>.png
#       reconstruction_log.csv
#       reconstruction_metrics.csv
#       metadata.json
#
# DO NOT TRAIN YET.
# ======================================================================================


# ======================================================================================
# 0. IMPORTS
# ======================================================================================

import json
import hashlib
import warnings

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from scipy.ndimage import gaussian_filter
from scipy.interpolate import PchipInterpolator

warnings.filterwarnings("ignore")


# ======================================================================================
# 1. PATHS
# ======================================================================================

OUT_DIR = Path("final_digitisation_v6")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GEOMETRY_PATH = Path(
    "manual_digitization_qc/manual_geometry.json"
)

V4_PATH = Path(
    "digitization_qc_v4/X_gaze_v4.npy"
)

DOWNLOADS = Path(
    "LOCAL_USER_HOME/Downloads"
)


# Find newest exported annotation JSON
candidates = list(
    DOWNLOADS.glob(
        "ecg_anchor_annotations*.json"
    )
)

if not candidates:

    raise FileNotFoundError(
        "Could not find ecg_anchor_annotations*.json "
        "in LOCAL_USER_HOME/Downloads/"
    )


ANNOTATION_PATH = max(
    candidates,
    key=lambda p: p.stat().st_mtime
)


print("=" * 100)
print("INPUTS")
print("=" * 100)

print("Annotations:", ANNOTATION_PATH)
print("Geometry:   ", GEOMETRY_PATH)
print("V4 array:   ", V4_PATH)


for p in [
    ANNOTATION_PATH,
    GEOMETRY_PATH,
    V4_PATH
]:

    if not p.exists():
        raise FileNotFoundError(p)


# ======================================================================================
# 2. LOAD
# ======================================================================================

with open(
    ANNOTATION_PATH,
    "r"
) as f:

    annotation_json = json.load(f)


with open(
    GEOMETRY_PATH,
    "r"
) as f:

    geometry = json.load(f)


X_v4 = np.load(
    V4_PATH
).astype(
    np.float32
)


assert X_v4.shape == (
    10,
    12,
    1000,
    1
)


X_final = X_v4.copy()


annotations = annotation_json.get(
    "annotations",
    {}
)


if not annotations:

    raise RuntimeError(
        "No annotations found in JSON."
    )


# ======================================================================================
# 3. FIXED ORDERS
# ======================================================================================

ECG_ORDER = [

    "NSR",
    "AFib",
    "Hyperkalemia",
    "AFlutter",
    "VTach",
    "WPW",
    "VentPaced",
    "LBBB",
    "STEMI",
    "CompleteHeartBlock",
]


LEAD_ORDER = [

    "I",
    "II",
    "III",

    "aVR",
    "aVL",
    "aVF",

    "V1",
    "V2",
    "V3",

    "V4",
    "V5",
    "V6",
]


DISPLAY_GRID = [

    ["I",   "aVR", "V1", "V4"],

    ["II",  "aVL", "V2", "V5"],

    ["III", "aVF", "V3", "V6"],
]


# ======================================================================================
# 4. CHECK JSON IS COMPLETE
# ======================================================================================

summary = []


for key, ann in annotations.items():

    summary.append({

        "ECG":
            ann.get("ecg"),

        "lead":
            ann.get("lead"),

        "status":
            ann.get("status"),

        "n_points":
            len(
                ann.get(
                    "points",
                    []
                )
            )
    })


summary_df = pd.DataFrame(
    summary
)


print("\nANNOTATION SUMMARY")
print(
    summary_df
    .sort_values(
        ["ECG", "lead"]
    )
    .to_string(
        index=False
    )
)


unfinished = summary_df[
    ~summary_df[
        "status"
    ].isin(
        [
            "corrected",
            "keep_v4"
        ]
    )
]


if len(unfinished) > 0:

    print(
        "\nUNFINISHED:"
    )

    print(
        unfinished.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Some annotations are still pending/editing."
    )


bad_corrected = summary_df[
    (
        summary_df[
            "status"
        ]
        ==
        "corrected"
    )
    &
    (
        summary_df[
            "n_points"
        ]
        <
        5
    )
]


if len(bad_corrected) > 0:

    raise RuntimeError(
        "A corrected lead has fewer than five anchors."
    )


print(
    "\n✓ Annotation JSON is complete."
)


# ======================================================================================
# 5. SOURCE IMAGE HELPERS
# ======================================================================================

def resolve_image(
    stored_path
):

    p = Path(
        stored_path
    )


    if p.exists():
        return p


    matches = list(
        Path(".").glob(
            f"**/{p.name}"
        )
    )


    if not matches:

        raise FileNotFoundError(
            f"Cannot locate {p.name}"
        )


    return matches[0]


def load_gray(
    path
):

    return (
        np.asarray(
            Image.open(
                path
            )
            .convert(
                "L"
            )
        )
        .astype(
            np.float32
        )
        /
        255.0
    )


# ======================================================================================
# 6. CONVERT BROWSER COORDINATES BACK TO ORIGINAL CROP
# ======================================================================================

def browser_points_to_crop(
    ann
):

    points = np.asarray(

        [
            [
                float(p["x"]),
                float(p["y"])
            ]
            for p in ann["points"]
        ],

        dtype=np.float32
    )


    sx = (
        float(
            ann[
                "original_crop_width"
            ]
        )
        /
        float(
            ann[
                "display_width"
            ]
        )
    )


    sy = (
        float(
            ann[
                "original_crop_height"
            ]
        )
        /
        float(
            ann[
                "display_height"
            ]
        )
    )


    points[:, 0] *= sx
    points[:, 1] *= sy


    # sort x ascending
    points = points[
        np.argsort(
            points[:, 0]
        )
    ]


    # Remove duplicate/near-duplicate x coordinates
    cleaned = [
        points[0]
    ]


    for p in points[1:]:

        if (
            p[0]
            -
            cleaned[-1][0]
            >=
            0.75
        ):

            cleaned.append(p)


    return np.asarray(
        cleaned,
        dtype=np.float32
    )


# ======================================================================================
# 7. IMAGE TRACE-EVIDENCE SCORE
# ======================================================================================

def ecg_score_map(
    gray
):

    gray = np.asarray(
        gray,
        dtype=np.float32
    )


    p1 = np.percentile(
        gray,
        1
    )

    p99 = np.percentile(
        gray,
        99
    )


    if p99 > p1 + 1e-8:

        norm = (
            gray
            -
            p1
        ) / (
            p99
            -
            p1
        )

    else:

        norm = gray.copy()


    norm = np.clip(
        norm,
        0,
        1
    )


    # Absolute darkness
    darkness = (
        1.0
        -
        norm
    )


    # Thin dark line vs local background
    fine = gaussian_filter(
        norm,
        sigma=0.45
    )


    background = gaussian_filter(
        norm,
        sigma=2.2
    )


    ridge = np.clip(
        background
        -
        fine,
        0,
        None
    )


    scale = np.percentile(
        ridge,
        99.5
    )


    if scale > 1e-8:

        ridge /= scale


    ridge = np.clip(
        ridge,
        0,
        1.5
    )


    score = (
        0.75
        *
        ridge
        +
        0.25
        *
        darkness
    )


    s = np.percentile(
        score,
        99.5
    )


    if s > 1e-8:

        score /= s


    return np.clip(
        score,
        0,
        1.5
    )


# ======================================================================================
# 8. ROBUST ANCHOR INTERPOLATION + LOCAL PIXEL SNAP
# ======================================================================================

def reconstruct_trace(
    crop,
    anchors
):

    """
    Manual anchors define waveform morphology.

    PCHIP provides the continuous manual trajectory.

    For each x-coordinate we only search a tiny vertical neighbourhood
    around that trajectory for actual dark ECG ink.

    There is NO global path optimisation, so reconstruction cannot become
    disconnected or fail with 'No valid final path'.
    """

    H, W = crop.shape


    anchors = np.asarray(
        anchors,
        dtype=np.float32
    )


    anchors[:, 0] = np.clip(
        anchors[:, 0],
        0,
        W - 1
    )


    anchors[:, 1] = np.clip(
        anchors[:, 1],
        0,
        H - 1
    )


    if len(anchors) < 5:

        raise RuntimeError(
            "Too few manual anchors."
        )


    # ------------------------------------------------------------------
    # First/last anchor delimit diagnostic lead duration
    # ------------------------------------------------------------------

    x_start = int(
        np.ceil(
            anchors[0, 0]
        )
    )


    x_end = int(
        np.floor(
            anchors[-1, 0]
        )
    )


    x_start = max(
        0,
        x_start
    )


    x_end = min(
        W - 1,
        x_end
    )


    if (
        x_end
        -
        x_start
        <
        20
    ):

        raise RuntimeError(
            "Annotated waveform span is too short."
        )


    xs = np.arange(
        x_start,
        x_end + 1,
        dtype=np.int32
    )


    # ------------------------------------------------------------------
    # PCHIP instead of ordinary cubic spline:
    #
    # shape-preserving and far less likely to overshoot between clicks.
    # ------------------------------------------------------------------

    interpolator = PchipInterpolator(
        anchors[:, 0],
        anchors[:, 1],
        extrapolate=False
    )


    guide = interpolator(
        xs
    ).astype(
        np.float32
    )


    # Numerical safety
    guide = np.clip(
        guide,
        0,
        H - 1
    )


    # ------------------------------------------------------------------
    # Local image snap
    # ------------------------------------------------------------------

    score = ecg_score_map(
        crop
    )


    # Small radius intentionally:
    # manual trajectory remains the dominant signal.
    snap_radius = max(
        3,
        int(
            round(
                0.025
                *
                H
            )
        )
    )


    # Never allow a huge snap.
    snap_radius = min(
        snap_radius,
        6
    )


    path = np.zeros(
        len(xs),
        dtype=np.float32
    )


    for i, x in enumerate(
        xs
    ):

        gy = float(
            guide[i]
        )


        centre = int(
            round(
                gy
            )
        )


        lo = max(
            0,
            centre - snap_radius
        )


        hi = min(
            H - 1,
            centre + snap_radius
        )


        ys = np.arange(
            lo,
            hi + 1
        )


        # --------------------------------------------------------------
        # Score favors dark ECG pixels but strongly penalises deviation
        # from manual guide.
        # --------------------------------------------------------------

        distance = np.abs(
            ys
            -
            gy
        )


        local_objective = (

            2.0
            *
            score[
                ys,
                x
            ]

            -

            0.22
            *
            distance
        )


        best = ys[
            np.argmax(
                local_objective
            )
        ]


        path[i] = float(
            best
        )


    # ------------------------------------------------------------------
    # FORCE the path particularly close to manually clicked anchors.
    # ------------------------------------------------------------------

    for ax, ay in anchors:

        nearest = int(
            np.argmin(
                np.abs(
                    xs
                    -
                    ax
                )
            )
        )


        if (
            0
            <=
            nearest
            <
            len(
                path
            )
        ):

            path[
                nearest
            ] = float(
                ay
            )


    # ------------------------------------------------------------------
    # Convert y-coordinate to ECG voltage orientation
    # ------------------------------------------------------------------

    signal = (
        -path
    )


    signal -= np.median(
        signal
    )


    # Resample to exactly 1000
    old_t = np.linspace(
        0,
        1,
        len(
            signal
        )
    )


    new_t = np.linspace(
        0,
        1,
        1000
    )


    signal_1000 = np.interp(
        new_t,
        old_t,
        signal
    ).astype(
        np.float32
    )


    signal_1000 -= np.mean(
        signal_1000
    )


    sd = np.std(
        signal_1000
    )


    if sd > 1e-8:

        signal_1000 /= sd


    # ------------------------------------------------------------------
    # QC metrics
    # ------------------------------------------------------------------

    deviation = np.abs(
        path
        -
        guide
    )


    y_idx = np.clip(
        np.round(
            path
        ).astype(
            int
        ),
        0,
        H - 1
    )


    path_scores = score[
        y_idx,
        xs
    ]


    return {

        "x":
            xs.astype(
                np.float32
            ),

        "y":
            path.astype(
                np.float32
            ),

        "guide":
            guide.astype(
                np.float32
            ),

        "anchors":
            anchors.astype(
                np.float32
            ),

        "signal":
            signal_1000.astype(
                np.float32
            ),

        "metrics": {

            "n_anchors":
                int(
                    len(
                        anchors
                    )
                ),

            "x_span":
                int(
                    len(
                        xs
                    )
                ),

            "snap_radius":
                int(
                    snap_radius
                ),

            "median_snap_distance":
                float(
                    np.median(
                        deviation
                    )
                ),

            "p95_snap_distance":
                float(
                    np.percentile(
                        deviation,
                        95
                    )
                ),

            "max_snap_distance":
                float(
                    np.max(
                        deviation
                    )
                ),

            "mean_path_score":
                float(
                    np.mean(
                        path_scores
                    )
                ),

            "median_path_score":
                float(
                    np.median(
                        path_scores
                    )
                ),
        }
    }


# ======================================================================================
# 9. RECONSTRUCT CORRECTED LEADS
# ======================================================================================

results = {}

log_rows = []

metric_rows = []


print(
    "\n" +
    "=" * 100
)

print(
    "RECONSTRUCTION"
)

print(
    "=" * 100
)


for key, ann in annotations.items():

    ecg = ann["ecg"]
    lead = ann["lead"]
    status = ann["status"]


    ecg_idx = ECG_ORDER.index(
        ecg
    )


    lead_idx = LEAD_ORDER.index(
        lead
    )


    if status == "keep_v4":

        log_rows.append({

            "ECG":
                ecg,

            "lead":
                lead,

            "status":
                "kept_v4",

            "n_anchors":
                0
        })


        print(
            f"{ecg:22s} {lead:4s} -> KEEP V4"
        )

        continue


    # ------------------------------------------------------------------
    # Source crop
    # ------------------------------------------------------------------

    box = [
        int(round(v))
        for v
        in ann["original_box"]
    ]


    x0, y0, x1, y1 = box


    source_path = resolve_image(
        geometry[
            ecg
        ][
            "image_path"
        ]
    )


    full_gray = load_gray(
        source_path
    )


    crop = full_gray[
        y0:y1,
        x0:x1
    ]


    anchors = browser_points_to_crop(
        ann
    )


    try:

        result = reconstruct_trace(
            crop,
            anchors
        )

    except Exception as e:

        print(
            f"\nERROR while reconstructing {ecg} {lead}:"
        )

        print(
            repr(e)
        )

        raise


    X_final[
        ecg_idx,
        lead_idx,
        :,
        0
    ] = result[
        "signal"
    ]


    results[
        (
            ecg,
            lead
        )
    ] = {

        "crop":
            crop,

        **result
    }


    log_rows.append({

        "ECG":
            ecg,

        "lead":
            lead,

        "status":
            "manual_anchor_reconstruction",

        "n_anchors":
            len(
                anchors
            )
    })


    metric_rows.append({

        "ECG":
            ecg,

        "lead":
            lead,

        **result[
            "metrics"
        ]
    })


    print(
        f"{ecg:22s} {lead:4s}"
        f" -> corrected ({len(anchors)} anchors)"
    )


print(
    "\n✓ All corrected leads reconstructed successfully."
)


# ======================================================================================
# 10. SAVE PATH FILES
# ======================================================================================

for (
    ecg,
    lead
), result in results.items():

    np.savez_compressed(

        OUT_DIR
        /
        f"PATH_{ecg}_{lead}.npz",

        x=result["x"],
        y=result["y"],
        guide=result["guide"],
        anchors=result["anchors"],
        signal=result["signal"],
    )


# ======================================================================================
# 11. SOURCE OVERLAY FIGURES
# ======================================================================================

print(
    "\nGenerating source-overlay QC..."
)


for ecg in ECG_ORDER:

    leads = [

        lead
        for lead
        in LEAD_ORDER

        if (
            ecg,
            lead
        )
        in
        results
    ]


    if not leads:
        continue


    n = len(
        leads
    )


    fig, axes = plt.subplots(

        n,
        2,

        figsize=(
            16,
            max(
                4,
                n * 3.2
            )
        ),

        squeeze=False
    )


    for row, lead in enumerate(
        leads
    ):

        result = results[
            (
                ecg,
                lead
            )
        ]


        crop = result[
            "crop"
        ]


        x = result[
            "x"
        ]


        y = result[
            "y"
        ]


        guide = result[
            "guide"
        ]


        anchors = result[
            "anchors"
        ]


        # LEFT
        ax = axes[
            row,
            0
        ]


        ax.imshow(
            crop,
            cmap="gray",
            origin="upper",
            aspect="auto"
        )


        ax.plot(
            x,
            guide,
            linestyle="--",
            linewidth=1.0,
            label="manual guide"
        )


        ax.plot(
            x,
            y,
            color="red",
            linewidth=1.5,
            label="final path"
        )


        ax.scatter(
            anchors[:, 0],
            anchors[:, 1],
            s=20,
            facecolors="none",
            edgecolors="yellow",
            linewidths=1,
            label="anchors"
        )


        ax.set_xlim(
            0,
            crop.shape[1] - 1
        )


        ax.set_ylim(
            crop.shape[0] - 1,
            0
        )


        ax.set_title(
            f"{lead}: source + final reconstruction"
        )


        ax.set_xticks([])
        ax.set_yticks([])


        if row == 0:

            ax.legend(
                fontsize=7,
                loc="upper right"
            )


        # RIGHT
        ax = axes[
            row,
            1
        ]


        lead_idx = LEAD_ORDER.index(
            lead
        )


        sig = X_final[
            ECG_ORDER.index(
                ecg
            ),
            lead_idx,
            :,
            0
        ]


        ax.plot(
            sig,
            linewidth=1
        )


        ax.axhline(
            0,
            linewidth=0.4,
            alpha=0.3
        )


        ax.set_title(
            f"{lead}: final pseudo-waveform"
        )


        ax.set_xlim(
            0,
            999
        )


        ax.set_xticks([])
        ax.set_yticks([])


    fig.suptitle(
        f"{ecg} — FINAL MANUAL-ANCHOR RECONSTRUCTION",
        fontsize=17,
        y=0.995
    )


    fig.tight_layout()


    fig.savefig(

        OUT_DIR
        /
        f"FINAL_OVERLAY_{ecg}.png",

        dpi=200,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ======================================================================================
# 12. FINAL 12-LEAD SIGNAL SHEETS
# ======================================================================================

print(
    "Generating 12-lead signal QC..."
)


for ei, ecg in enumerate(
    ECG_ORDER
):

    fig, axes = plt.subplots(

        3,
        4,

        figsize=(
            18,
            10
        ),

        constrained_layout=True
    )


    for r in range(
        3
    ):

        for c in range(
            4
        ):

            lead = DISPLAY_GRID[
                r
            ][
                c
            ]


            li = LEAD_ORDER.index(
                lead
            )


            sig = X_final[
                ei,
                li,
                :,
                0
            ]


            ax = axes[
                r,
                c
            ]


            ax.plot(
                sig,
                linewidth=1
            )


            ax.axhline(
                0,
                linewidth=0.4,
                alpha=0.3
            )


            key = (
                ecg
                +
                "::"
                +
                lead
            )


            suffix = ""


            if key in annotations:

                status = annotations[
                    key
                ][
                    "status"
                ]


                if status == "corrected":

                    suffix = " [MANUAL]"


                elif status == "keep_v4":

                    suffix = " [V4 REVIEWED]"


            ax.set_title(
                lead
                +
                suffix
            )


            ax.set_xlim(
                0,
                999
            )


            ax.set_xticks([])
            ax.set_yticks([])


    fig.suptitle(
        f"{ecg} — FINAL candidate pseudo-waveforms",
        fontsize=18
    )


    fig.savefig(

        OUT_DIR
        /
        f"FINAL_SIGNAL_{ecg}.png",

        dpi=200,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ======================================================================================
# 13. SAVE FINAL ARRAY
# ======================================================================================

FINAL_ARRAY_PATH = (
    OUT_DIR
    /
    "X_gaze_final.npy"
)


assert X_final.shape == (
    10,
    12,
    1000,
    1
)


if not np.all(
    np.isfinite(
        X_final
    )
):

    raise RuntimeError(
        "Final array contains non-finite values."
    )


np.save(
    FINAL_ARRAY_PATH,
    X_final
)


# ======================================================================================
# 14. LOGS
# ======================================================================================

log_df = pd.DataFrame(
    log_rows
)


metrics_df = pd.DataFrame(
    metric_rows
)


log_df.to_csv(

    OUT_DIR
    /
    "reconstruction_log.csv",

    index=False
)


metrics_df.to_csv(

    OUT_DIR
    /
    "reconstruction_metrics.csv",

    index=False
)


print(
    "\nRECONSTRUCTION LOG"
)

print(
    log_df
    .sort_values(
        ["ECG", "lead"]
    )
    .to_string(
        index=False
    )
)


print(
    "\nCORRECTED-LEAD METRICS"
)


print(
    metrics_df[
        [
            "n_anchors",
            "median_snap_distance",
            "p95_snap_distance",
            "max_snap_distance",
            "mean_path_score",
            "median_path_score"
        ]
    ]
    .describe()
    .to_string()
)


# ======================================================================================
# 15. HASH / METADATA
# ======================================================================================

def sha256(
    path
):

    h = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                1024 * 1024
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


final_hash = sha256(
    FINAL_ARRAY_PATH
)


metadata = {

    "created":
        datetime.now().isoformat(),

    "annotation_json":
        str(
            ANNOTATION_PATH
        ),

    "source_v4":
        str(
            V4_PATH
        ),

    "geometry":
        str(
            GEOMETRY_PATH
        ),

    "final_array":
        str(
            FINAL_ARRAY_PATH
        ),

    "final_sha256":
        final_hash,

    "shape":
        list(
            X_final.shape
        ),

    "ecg_order":
        ECG_ORDER,

    "lead_order":
        LEAD_ORDER,

    "reconstruction": (
        "Shape-preserving PCHIP interpolation through manually "
        "placed waveform anchors followed by local image-based "
        "vertical snapping within <=6 pixels. No global dynamic "
        "programming was used."
    ),

    "training_performed":
        False
}


with open(

    OUT_DIR
    /
    "metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )


# ======================================================================================
# 16. FINISH
# ======================================================================================

print(
    "\n" +
    "=" * 100
)

print(
    "V6 RECONSTRUCTION COMPLETE"
)

print(
    "=" * 100
)


print(
    "\nFinal candidate:"
)

print(
    FINAL_ARRAY_PATH
)


print(
    "\nShape:",
    X_final.shape
)


print(
    "\nSHA256:",
    final_hash
)


print(
    """
NOW INSPECT:

    final_digitisation_v6/FINAL_OVERLAY_*.png

The decisive visual elements are:

    yellow circles = your manual points
    dashed line    = interpolation through your points
    red line       = locally image-snapped final path

The red line should remain very close to the dashed guide AND lie on
the black ECG waveform.

Also inspect:

    final_digitisation_v6/FINAL_SIGNAL_*.png

DO NOT START MODEL TRAINING YET.
"""
)